# Whisper, WER y EnCodec

Notebook opcional del **módulo 16** del curso (en la app: `uv run dev` y la ruta `#/modulo/audio-transformers`). Usa modelos reales; el módulo interactivo explica la teoría con NumPy.

> **Requisitos:** desde la raíz del repositorio, `uv sync --group notebooks` y abre este notebook con `uv run --group notebooks jupyter lab`. La primera ejecución descarga los modelos de Hugging Face.

In [ ]:
import jiwer
import torch
from datasets import load_dataset
from transformers import pipeline

ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
asr = pipeline("automatic-speech-recognition", model="openai/whisper-small")

In [ ]:
refs, hips = [], []
for ej in ds.select(range(5)):
    hip = asr(ej["audio"]["array"].copy(), generate_kwargs={"language": "english", "task": "transcribe"})["text"]
    refs.append(ej["text"].lower())
    hips.append(hip.lower().replace(",", "").replace(".", ""))
print(hips[0])
print("WER:", round(jiwer.wer(refs, hips), 3))

In [ ]:
print(asr(ds[0]["audio"]["array"].copy(), generate_kwargs={"language": "english", "task": "translate"}, return_timestamps=True))

Tokens de audio con EnCodec: menos ancho de banda = menos libros de códigos.

In [ ]:
from transformers import AutoProcessor, EncodecModel

modelo = EncodecModel.from_pretrained("facebook/encodec_24khz")
proc = AutoProcessor.from_pretrained("facebook/encodec_24khz")
import librosa

audio = librosa.resample(ds[0]["audio"]["array"], orig_sr=16000, target_sr=24000)
x = proc(raw_audio=audio, sampling_rate=24000, return_tensors="pt")
for kbps in [1.5, 6.0, 24.0]:
    cod = modelo.encode(x["input_values"], x["padding_mask"], bandwidth=kbps)
    print(kbps, "kbps -> códigos", tuple(cod.audio_codes.shape))